# 11 — Scattering Transforms

**Purpose:** Compute scattering transform coefficients for CIB and tSZ patches
and compare Agora, DDPM, and Gaussian baseline distributions.

The scattering transform is a non-linear multi-scale statistical descriptor that
captures information complementary to the power spectrum. It applies a sequence
of wavelet convolutions and modulus operations, producing coefficients that are
sensitive to non-Gaussian structure at multiple angular scales and orientations.

This notebook computes:
- **S1** — first-order coefficients: mean wavelet modulus at each scale j and
  orientation l. These are related to the power spectrum.
- **S2** — second-order coefficients: mean modulus of modulus at pairs (j1,l1),
  (j2,l2) with j2 > j1. These capture non-Gaussian cross-scale correlations.
- **C11** (optional) — scattering covariance: full cross-scale correlation
  matrix. Requires the Cheng et al. backend.

## Installation

**Option A — Cheng et al. (recommended, faster, exposes C11):**
```bash
cd ~/cmb_foregrounds_diffusion
git clone https://github.com/SihaoCheng/scattering_transform.git
```
Then add to sys.path in this notebook (see cell below).

**Option B — kymatio (pip-installable, slower, no C11):**
```bash
pip install kymatio
```

**Inputs:**
- Test maps: `data/low_pass/2mJy/*.npy`
- DDPM samples: `data/low_pass/2mJy/new_samples_*.npy`
- Norm params: `data/low_pass/2mJy/norm_params_2mJy.npy`

**Outputs:** `plots/scattering_coefficients.pdf`

**Key module functions:**
- `foregrounds_diffusion.scattering_stats.compute_scattering_coefficients`
- `foregrounds_diffusion.scattering_stats.compute_scattering_covariance`

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# If using Cheng et al. backend, add the cloned repo to the path
SCATTERING_REPO = Path('/home/apb86/cmb_foregrounds_diffusion/scattering_transform')
if SCATTERING_REPO.exists():
    sys.path.insert(0, str(SCATTERING_REPO))
    print('Using Cheng et al. scattering backend')
else:
    print('Cheng et al. repo not found — will try kymatio')
    print(f'To install: git clone https://github.com/SihaoCheng/scattering_transform.git {SCATTERING_REPO}')

from foregrounds_diffusion.scattering_stats import (
    compute_scattering_coefficients,
    compute_scattering_covariance,
    scattering_summary,
)

PTSRC            = 2
PIXEL_RES_ARCMIN = 1.40625
N_MAPS           = 100    # number of maps per source
J                = 5      # number of scales: 2^1 … 2^5 pixels
L                = 4      # number of orientations

PATCHES_DIR  = Path(f'data/low_pass/{PTSRC}mJy')
PROJECT_ROOT = Path('/home/apb86/cmb_foregrounds_diffusion')


In [ ]:
# ---------------------------------------------------------------------------
# Load and denormalise maps
# ---------------------------------------------------------------------------
norm_params = np.load(PATCHES_DIR / f'norm_params_{PTSRC}mJy.npy')
cib_mean, cib_std, tsz_mean, tsz_std = norm_params

cib_maps = np.load(PATCHES_DIR / f'CIB_map_150GHz_256_st6_zscore_{PTSRC}mJy_lp.npy')
tsz_maps = np.load(PATCHES_DIR / f'tSZ3_map_150GHz_256_st6_zscore_{PTSRC}mJy_lp.npy')

rng = np.random.default_rng(seed=42)
indices  = rng.permutation(len(cib_maps))
test_idx = indices[int(0.8 * len(cib_maps)):]

cib_test = (cib_maps[test_idx, :, :, 0] * cib_std + cib_mean).astype(np.float32)
tsz_test = (tsz_maps[test_idx, :, :, 0] * tsz_std + tsz_mean).astype(np.float32)
print(f'Test patches: {len(cib_test)}')

ddpm_raw = np.load(
    PROJECT_ROOT / 'data' / 'low_pass' / f'{PTSRC}mJy' /
    f'new_samples_cib_tsz_{PTSRC}mJy_zero_norm_6x6_w_au_lp.npy'
)
ddpm_cib = (ddpm_raw[:, 0] * cib_std + cib_mean).astype(np.float32)
ddpm_tsz = (ddpm_raw[:, 1] * tsz_std + tsz_mean).astype(np.float32)

gauss_maps = np.load(PATCHES_DIR / f'gaussian_cib_tsz_{PTSRC}mJy_lp.npy')
gauss_cib  = (gauss_maps[:, 0] if gauss_maps.shape[1] == 2
              else gauss_maps[:, :, :, 0]).astype(np.float32)
gauss_tsz  = (gauss_maps[:, 1] if gauss_maps.shape[1] == 2
              else gauss_maps[:, :, :, 1]).astype(np.float32)

N = min(N_MAPS, len(cib_test), len(ddpm_cib), len(gauss_cib))
print(f'Using {N} maps per source')


In [ ]:
# ---------------------------------------------------------------------------
# Compute scattering coefficients
# NOTE: ~2-5 min per source on CPU; much faster on GPU
# ---------------------------------------------------------------------------
print('Computing Agora CIB scattering coefficients...')
s_agora_cib = compute_scattering_coefficients(cib_test[:N], J=J, L=L)

print('Computing Agora tSZ scattering coefficients...')
s_agora_tsz = compute_scattering_coefficients(tsz_test[:N], J=J, L=L)

print('Computing DDPM CIB scattering coefficients...')
s_ddpm_cib = compute_scattering_coefficients(ddpm_cib[:N], J=J, L=L)

print('Computing DDPM tSZ scattering coefficients...')
s_ddpm_tsz = compute_scattering_coefficients(ddpm_tsz[:N], J=J, L=L)

print('Computing Gaussian CIB scattering coefficients...')
s_gauss_cib = compute_scattering_coefficients(gauss_cib[:N], J=J, L=L)

print('Computing Gaussian tSZ scattering coefficients...')
s_gauss_tsz = compute_scattering_coefficients(gauss_tsz[:N], J=J, L=L)

print('Done.')


In [ ]:
# ---------------------------------------------------------------------------
# Plot S1 coefficients averaged over orientations
# S1[j] = mean wavelet modulus at scale 2^j — related to power spectrum
# S1 shape: (N, J) — already orientation-averaged (S1_iso from Cheng et al.)
# ---------------------------------------------------------------------------
scales = np.arange(1, J + 1)  # j = 1 … J
scale_labels = [f'$2^{j}$ px' for j in scales]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sources_s1 = [
    ('Agora',    'C0', '-',  s_agora_cib, s_agora_tsz),
    ('DDPM',     'C1', '-',  s_ddpm_cib,  s_ddpm_tsz),
    ('Gaussian', 'C2', '--', s_gauss_cib, s_gauss_tsz),
]

for ax, title, idx in zip(axes, ['CIB', 'tSZ'], [2, 3]):
    for label, color, ls, s_cib, s_tsz in sources_s1:
        s = s_cib if title == 'CIB' else s_tsz
        # S1 shape: (N, J) — no orientation axis to average over
        s1 = s['S1']           # (N, J)
        mean = s1.mean(axis=0) # (J,)
        std  = s1.std(axis=0)
        ax.plot(scales, mean, color=color, ls=ls, lw=1.5, label=label, marker='o')
        ax.fill_between(scales, mean - std, mean + std, alpha=0.2, color=color)
    ax.set_xticks(scales)
    ax.set_xticklabels(scale_labels)
    ax.set_xlabel('Wavelet scale j')
    ax.set_ylabel(r'$S_1(j)$ (orientation-averaged)')
    ax.set_title(f'{title} — First-order scattering coefficients S1')
    ax.legend()
    ax.set_yscale('log')

plt.tight_layout()
Path('plots').mkdir(exist_ok=True)
plt.savefig('plots/scattering_S1.pdf', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Plot S2 coefficients — cross-scale coupling matrix
# S2 shape: (N, J, J, L) — axes are (map, j1, j2, orientation_difference)
# Average over N and orientation difference to get a (J, J) matrix
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, title, s_agora, s_ddpm in zip(
    axes,
    ['CIB', 'tSZ'],
    [s_agora_cib, s_agora_tsz],
    [s_ddpm_cib,  s_ddpm_tsz],
):
    # S2 shape: (N, J, J, L) → average over N and L → (J, J)
    s2_agora = s_agora['S2'].mean(axis=(0, 3))  # (J, J)
    s2_ddpm  = s_ddpm['S2'].mean(axis=(0, 3))

    # Ratio DDPM/Agora (only upper triangle where j2>j1 is defined)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(s2_agora > 0, s2_ddpm / s2_agora, np.nan)

    im = ax.imshow(ratio, origin='lower', cmap='RdBu_r',
                   vmin=0.5, vmax=1.5, aspect='auto')
    ax.set_xticks(range(J)); ax.set_yticks(range(J))
    ax.set_xticklabels([f'j={j+1}' for j in range(J)])
    ax.set_yticklabels([f'j={j+1}' for j in range(J)])
    ax.set_xlabel('Scale j2'); ax.set_ylabel('Scale j1')
    ax.set_title(f'{title} — S2 ratio DDPM/Agora\n(1=perfect agreement)')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('plots/scattering_S2_ratio.pdf', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Summary: flattened scattering feature vector residuals
# ---------------------------------------------------------------------------
feat_agora_cib = scattering_summary(s_agora_cib)  # (N, n_features)
feat_ddpm_cib  = scattering_summary(s_ddpm_cib)
feat_agora_tsz = scattering_summary(s_agora_tsz)
feat_ddpm_tsz  = scattering_summary(s_ddpm_tsz)

resid_cib = ((feat_agora_cib.mean(0) - feat_ddpm_cib.mean(0))
             / (feat_agora_cib.std(0) + 1e-10))
resid_tsz = ((feat_agora_tsz.mean(0) - feat_ddpm_tsz.mean(0))
             / (feat_agora_tsz.std(0) + 1e-10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, resid, title in zip(axes, [resid_cib, resid_tsz], ['CIB', 'tSZ']):
    ax.bar(range(len(resid)), resid, color='C0', alpha=0.7)
    ax.axhline(0,  color='k',    lw=0.8, ls='--')
    ax.axhline(1,  color='gray', lw=0.6, ls=':')
    ax.axhline(-1, color='gray', lw=0.6, ls=':')
    ax.set_xlabel('Scattering feature index')
    ax.set_ylabel(r'$(\bar{S}^{\rm Agora} - \bar{S}^{\rm DDPM}) / \sigma^{\rm Agora}$')
    ax.set_title(f'{title} scattering coefficient residuals')
    ax.set_ylim(-3, 3)

plt.tight_layout()
plt.savefig('plots/scattering_residuals.pdf', dpi=200, bbox_inches='tight')
plt.show()

print(f'CIB: {(np.abs(resid_cib) < 1).mean():.1%} of features within 1σ')
print(f'tSZ: {(np.abs(resid_tsz) < 1).mean():.1%} of features within 1σ')


In [ ]:
# ---------------------------------------------------------------------------
# Optional: scattering covariance C11 (Cheng et al. backend only)
# ---------------------------------------------------------------------------
print('Computing scattering covariance C11 for Agora CIB...')
cov_agora_cib = compute_scattering_covariance(cib_test[:N], J=J, L=L)

if cov_agora_cib is not None:
    print(f'Available keys: {list(cov_agora_cib.keys())}')
    c11_iso = cov_agora_cib['C11_iso']  # shape (N, J, J, J, L, L)
    print(f'C11_iso shape: {c11_iso.shape}')

    # Mean C11_iso over orientations and maps
    c11_mean = np.nanmean(np.abs(c11_iso), axis=(0, 4, 5))  # (J, J, J)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(c11_mean.mean(axis=0), origin='lower', cmap='viridis')
    ax.set_xlabel('Scale j2'); ax.set_ylabel('Scale j3')
    ax.set_title('Agora CIB — mean |C11_iso| (j1-averaged)')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig('plots/scattering_C11.pdf', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('Scattering covariance not available (requires Cheng et al. backend).')
